In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import torch
from torch.utils.data import DataLoader
import hydra
from omegaconf import OmegaConf
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from src.activations import ActivationDataset
from src.sae import SparseAutoencoder
from src.interpreter import SAEInterpreter
from src.utils import load_sae

from global_settings import settings

# 기본 세팅

In [ ]:
# ======================================== FONT SETTING ========================================
import matplotlib
from matplotlib import font_manager

font_path = './assets/NanumGothic.ttf'
font_manager.fontManager.addfont(font_path)
font = font_manager.FontProperties(fname=font_path).get_name()
matplotlib.rc('font', family=font)
matplotlib.rcParams['font.family'] ='NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] =False
# ==============================================================================================


# ======================================== BASCI SETTING ========================================
BATCH_SIZE = 128
MAX_SEQ_LEN = settings.prev_seq_len  # 1024
assert MAX_SEQ_LEN == 1024, "MAX_SEQ_LEN must be 1024"
# ==============================================================================================


# ======================================== DATA & MODEL ========================================
tokenizer = AutoTokenizer.from_pretrained(str(settings.lm_weights_path))

text_df = pd.read_csv(str(settings.text_dataset_path))
print(f"text_df.shape: {text_df.shape}")
print(f"text_df.columns: {text_df.columns.tolist()}")

MODEL_TAG = "f3072_k256"
LAYER_IDX = 21
RUN_TAG = f"{MODEL_TAG}_alpha_L{LAYER_IDX}"

SAE_MODEL_PATH = f"./outputs/weights/{RUN_TAG}"
sae, sae_ckpt = load_sae(SAE_MODEL_PATH, return_ckpt=True)
sae.to(torch.device("cuda"))
print(f"Loaded SAE model's checkpoint: {sae_ckpt}")

activations_dir = f"./data/post_mlp_residual_stream/layer_{LAYER_IDX}"
dataset = ActivationDataset(data_dir=activations_dir, dtype=torch.float32)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
# ==============================================================================================

**테스트 샘플**

In [ ]:
samples, mask, sample_idxs = next(iter(dataloader))

idx = 4
sample_ts = samples[idx]
sample_mask = mask[idx]
sample_idx = sample_idxs[idx].item()
sample_text = text_df.iloc[sample_idx]["text"]
print("TEXT: ", sample_text)
print("TENSOR: ", sample_ts.shape)

In [ ]:
device = torch.device("cuda")

In [ ]:
x = sample_ts.unsqueeze(0).to(torch.device("cuda"))
preprocessed_input = sae._preprocess_input(x)
encoded = sae.encoder(preprocessed_input)
features = sae._apply_activation(encoded)

encoded_flat = encoded.view(-1).cpu().detach().numpy()
features_flat = features.view(-1).cpu().detach().numpy()

### **검증-1: token/sample 별 feature의 활성화 점수를 확인해보자.**

In [ ]:
samples, mask, sample_idxs = next(iter(dataloader))

idx = 4
sample_ts = samples[idx]
sample_idx = sample_idxs[idx].item()
sample_text = text_df.iloc[sample_idx]["text"]

x = sample_ts.unsqueeze(0).to(torch.device("cuda"))
preprocessed_input = sae._preprocess_input(x)
encoded = sae.encoder(preprocessed_input)
features = sae._apply_activation(encoded)

In [ ]:
# 랜덤 feature 10개 '토큰' 점수 분포 확인해보기
token_scores_flat = []
for f_idx in range(features.shape[-1])[:10]:
    activations = features[0, :, f_idx]
    token_scores = activations.to("cpu").detach().numpy().tolist()
    token_scores_flat.extend(token_scores)

In [ ]:
plt.plot(token_scores_flat)
plt.show()

### **검증-2: 토큰 활성화 점수를 계산할 때 encoded / features 중 어느 것을 사용해야 하는지 확인 필요:**
- encoded: `encoded = self.encoder(processed_input)`
- features: `features=_apply_activation(encoded)`

In [ ]:
plt.figure()
# plt.hist(features_flat, bins=500, alpha=0.5, label='Features', density=True)
plt.hist(encoded_flat, bins=500, alpha=0.5, label='Encoded', density=True)
# plt.title("'Feature Score' Distribution (before activation vs after activation)")
plt.title("'Feature Score' Distribution (before activation)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
plt.figure()
features_flat_non_zero = features_flat[features_flat != 0]  # Remove zeros for better visualization
plt.hist(features_flat, bins=500, alpha=0.5, label='Features', density=True)
plt.hist(features_flat_non_zero, bins=500, alpha=0.5, label='Features(nz)', density=True)
plt.title("'Feature Score' Distribution (after activation)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
plt.figure()
features_flat_non_zero = features_flat[features_flat != 0]  # Remove zeros for better visualization
plt.hist(features_flat_non_zero, bins=500, alpha=0.5, label='Features(nz)', density=True)
plt.title("'Feature Score' Distribution (after activation) (non-zero)")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.show()

### **검증-3: Feature 벡터 자체의 Density (encoder/decoder 형태가 sparse or dense?)**

In [ ]:
w_enc = sae.encoder.weight.detach()
w_dec = sae.decoder.weight.detach()
print(w_enc.shape)  # torch.Size([3072, 768])
print(w_dec.shape)  # torch.Size([768, 3072])

feat_vec = w_enc[:, FEATURE_TO_ANALYZE:=222]
feat_vec_ = w_dec[FEATURE_TO_ANALYZE, :]
print(feat_vec.shape)  # torch.Size([3072])
print(feat_vec_.shape)  # torch.Size([3072])

In [ ]:
feat_vec

In [ ]:
feat_vec_norms = []
feat_vec_norms_ = []

for i in range(w_enc.shape[0]):
    vec = w_enc[i]
    vec_ = w_dec.T[i]
    norm = vec.abs().sum().detach().cpu().item()
    norm_ = vec_.abs().sum().detach().cpu().item()
    feat_vec_norms.append(norm)
    feat_vec_norms_.append(norm_)

In [ ]:
plt.figure()
plt.title("L1 Norm Distribution of Weight Vectors")

# plt.hist(feat_vec_norms, bins=500, alpha=0.5, density=True, label="encoder")
plt.plot(feat_vec_norms, label="encoder")
# plt.hist(feat_vec_norms_, bins=500, alpha=0.5, density=True, label="decoder")
plt.plot(feat_vec_norms_, label="decoder")
# plt.xlabel("L1 Norm")
# plt.ylabel("Density")
plt.xlabel("Neuron Index")
plt.ylabel("Value")

plt.legend()
plt.show()

### **검증-4: Feature가 활성화 시키는 토큰들의 density**
- 여기서 **Density**는 앞에서와 다르게, 모델이 입력을 처리할 때 **평균적으로 몇 개의 Feature를 활성화시키는지**를 의미. (L0 Norm)
- **Top-K SAE의 경우, 이 값은 k와 거의 일치해야 함.**

In [ ]:
avg_densities = []
text_indices = []

with torch.no_grad():
    for batch in tqdm(dataloader):
        batch, mask, text_ids = batch
        text_indices.extend(text_ids.numpy().tolist())
        features, _, _ = sae(batch.to(torch.device("cuda")))
        l0_norms = torch.count_nonzero(features, dim=-1)
        avg_density = l0_norms.float().mean()
        avg_densities.append(avg_density.detach().cpu().item())

In [ ]:
plt.figure()
plt.plot(avg_densities)
plt.show()

In [ ]:
avg_density, l0_norms

### Token-Feature 활성화 예시

In [ ]:
random_batch_idx = 3

num_tokens_to_show = 64
num_features_to_show = 64

for batch_idx, (samples, masks, indices) in enumerate(dataloader):
    if batch_idx == random_batch_idx:
        sample = samples[0]
        sample_idx = indices[0].item()
        break

sample_text = text_df.iloc[sample_idx]["text"]
tokens = tokenizer.tokenize(sample_text, truncation=True, max_length=MAX_SEQ_LEN)
sample = sample[:len(tokens)]
features = interpreter._get_feature_activations(sample.unsqueeze(0).to(torch.device("cuda")))
features = features[0].detach().cpu().numpy()

num_tokens, num_features = features.shape

# 활성화 토큰 span 자동 탐색 (window 슬라이딩으로 가장 활성화된 구간 찾기)
activation_mask = (features > 0).astype(int)
token_activation_counts = activation_mask.sum(axis=1)  # 각 토큰별 활성화된 feature 개수

best_start = 0
best_sum = -1
for i in range(0, num_tokens - num_tokens_to_show + 1):
    window_sum = token_activation_counts[i:i + num_tokens_to_show].sum()
    if window_sum > best_sum:
        best_sum = window_sum
        best_start = i

best_token_span = (best_start, best_start + num_tokens_to_show)
span_features = features[best_token_span[0]:best_token_span[1], :]

# 활성화 수가 많은 feature 상위 N개를 찾음
feature_activation_counts = (span_features > 0).sum(axis=0)
top_feature_indices_unsorted = np.argsort(feature_activation_counts)[-num_features_to_show:]
top_feature_indices = np.sort(top_feature_indices_unsorted)

data_to_plot = span_features[:, top_feature_indices]

span_tokens = tokens[best_token_span[0]:best_token_span[1]]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(data_to_plot, cmap="viridis", cbar=True, ax=ax)

ax.set_yticks(np.arange(len(span_tokens)))
ax.set_yticklabels(span_tokens, rotation=0, fontsize=8)

ax.set_title(f"Feature Activations (Sample {sample_idx})", fontsize=16)
ax.set_xlabel("Top Activated Features", fontsize=12)
ax.set_ylabel("Tokens", fontsize=12)

plt.tight_layout()
plt.show()

### 샘플 score

In [ ]:
interpreter.plot_scores(results, plot_scope="sample", feature_idx=FEATURE_TO_ANALYZE)

### 토큰 score

In [ ]:
interpreter.plot_scores(results, plot_scope="token")

## 특징 해석 PoC

In [ ]:
TOP_K = 20
CONTEXT_WINDOW = 100
top_k_activations = TopK(k=TOP_K)
global_token_index = 0
batch_size = analysis_loader.batch_size
_device = sae_model.encoder.weight.device

for batch_idx, activations_batch in enumerate(tqdm(analysis_loader)):
    activations_batch = activations_batch.to(_device)

    with torch.no_grad():
        processed_activations = sae_model._preprocess_input(activations_batch)
        encoded = sae_model.encoder(processed_activations)
        features = sae_model._apply_activation(encoded)  # (batch_size, seq_len, feature_dim)
        
        target_feature_activations = features[..., FEATURE_TO_ANALYZE]  # (batch_size, seq_len)
        
        #if target_feature_activations.max() > 0:
        for i in range(activations_batch.shape[0]):
            try:
                text_idx = batch_idx * batch_size + i  # Verify that activation indices match text indices.
                text = text_df.iloc[text_idx]["text"]
                tokens = tokenizer.tokenize(text)

                max_val, max_idx = torch.max(target_feature_activations[i], dim=0)  # NOTE: why not topk?

                if max_val > 0:
                    token_index = max_idx.item()
                    activating_token = tokens[token_index]

                    start = max(0, token_index - CONTEXT_WINDOW)
                    end = min(len(tokens), token_index + CONTEXT_WINDOW + 1)
                    snippet_tokens = tokens[start:end]
                    snippet_text = tokenizer.convert_tokens_to_string(snippet_tokens)

                    top_k_activations.push(max_val.item(), snippet_text, activating_token)
            
            except IndexError:
                break

In [ ]:
for value, snippet, token in top_k_activations.top_results():
    highlighted_snippet = snippet.replace(token, f"\033[1;31m{token}\033[0m")
    print(f"Activation: {value:.4f} | Token: '{token}'")
    print(f"  Snippet: {highlighted_snippet}")
    print("-" * 20)